# VKRR: learning and evaluation — Section 6.2, Figures 2–4 and Table 1

Vector-valued kernel ridge regression with separable kernels $K = k\,I_{\mathcal Y}$ on
the data from notebook 02.

## Correspondence to the kernel definitions in the paper

The paper writes the two isotropic kernels as

$$k_{\mathrm{Gauss}}(x,x')=\exp(-\bar s^{\,2}\|x-x'\|^2),\qquad
k_{\mathrm{Sob}}(x,x')=\frac{2^{1-\bar r}}{\Gamma(\bar r)}
\bigl(\sqrt{2\bar r}\,\bar s\|x-x'\|\bigr)^{\bar r}
\mathcal{K}_{\bar r}\bigl(\sqrt{2\bar r}\,\bar s\|x-x'\|\bigr),$$

and `gauss(x, y, s)` / `matern52(x, y, s)` below implement exactly these with
$\bar s = s$: for $\bar r = 5/2$ the closed form of the Matérn kernel is
$(1+\sqrt5\,\bar s d+\tfrac53\bar s^{\,2}d^{2})e^{-\sqrt5\,\bar s d}$ and
$\sqrt5=\sqrt{2\bar r}$. **The entries of `S_GRID` are therefore literally the
$\bar s$ values reported in Section 6.2**, no conversion needed.
(Notebook 01 uses scikit-learn, whose `length_scale` translates differently; see the
header there.)

## Protocol

* **Szegő kernel** with the theory-adapted parameters
  $c_p = c_{\mathrm{scale}}\cdot 0.999^2\,p^{-2/5}$. Assumption 6 requires
  $c_p > \rho_p^2$ for some $\delta$-admissible $(\rho_p)$; rescaling
  $\rho_p = \sqrt{c_{\mathrm{scale}}}\cdot0.999\,p^{-1/5}$ shows that the family is
  covered by Theorem 13 for $c_{\mathrm{scale}} > 0.928$ (sharp bound) resp.
  $> 0.991$ (crude bound) — see the verification cell in notebook 02. Of the grid
  below **only $c_{\mathrm{scale}}=0.99$ is admissible**; the smaller values are
  included to probe the sensitivity to this parameter, not as theory-covered choices.
* **Isotropic baselines:** Matérn $\bar r = 5/2$ and Gaussian, both tuned over
  `S_GRID`. `S_GRID` and `C_GRID` are the **shared** grids of eq. (25): the scalar
  experiments of Section 6.1 use exactly the same two grids, so both experiments
  follow one protocol. The cross-validated optimum is interior in both.
* **Model selection:** per repetition 5-fold CV over kernel parameter × nugget
  $\{10^{-3},10^{-5},10^{-7},10^{-9}\}$. The nugget is added directly to $K$ and hence equals
  $N\lambda$ in the ERM normalisation (28) of the paper — Section 6.2 states it in
  this form. The kernel matrix is built once per parameter and sliced for all
  folds/nuggets (numerically identical, ~3× faster).
* **Evaluation:** predictions are $P_1$ functions on the training mesh, embedded into
  the $320\times320$ reference mesh by **bilinear interpolation** — error order
  $O(h^2)$, the same as the FEM itself (nearest-neighbour would be $O(h)$ and
  dominates every other error source). Relative $L^2(\Omega)$ errors via the
  trapezoidal rule; mean over the 1000 test samples.
* **Pipeline floor:** the same test parameters solved directly on the training mesh —
  the error under perfect learning; no kernel can be expected to beat it.
* float64 in JAX throughout (float32 Cholesky is unreliable for the ill-conditioned
  Szegő matrices).
* Significance: Welch tests at level $0.01$ over the 10 independent repetitions;
  the $\pm$ values in Table 1 are `np.std`, i.e. with $\mathrm{ddof}=0$.
* **Oracle sweep:** the last cells reproduce the claim in Section 6.2 that
  cross-validation attains the achievable optimum, by minimising the *test* error
  over substantially denser parameter grids.

Runtime: $N$-experiment ~1–2 h, robustness run ~10 min, $P$-experiment ~2 h,
oracle sweep ~2–3 h.

In [ ]:
import time
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)      # essential for the Szego matrices
import jax.numpy as jnp
from jax.scipy.linalg import cho_factor, cho_solve
from sklearn.model_selection import KFold
from scipy.sparse import csr_matrix
from scipy.stats import ttest_ind, linregress
import matplotlib.pyplot as plt


# ----------------------------- kernels -----------------------------
def szego(x, y, c_scale, P):
    """Szego product kernel with adapted c_p = c_scale * 0.999^2 * p^{-2/5}."""
    p = jnp.arange(1, P + 1)
    c = c_scale * 0.999 ** 2 * p ** (-2 / 5)
    return jnp.prod(1.0 / (1.0 - c * x[:, None, :] * y[None, :, :]), axis=2)


def _sqdist(x, y):
    return jnp.clip(jnp.sum(x**2, 1)[:, None] + jnp.sum(y**2, 1)[None, :]
                    - 2 * x @ y.T, 0.0)


def matern52(x, y, s):
    """Matern with rbar = 5/2 in the normalisation of eq. (24): the argument is
    sqrt(2*rbar) * sbar * d = sqrt(5) * s * d, i.e. s is exactly the paper's sbar."""
    d = jnp.sqrt(_sqdist(x, y))
    return (1 + jnp.sqrt(5)*d*s + 5*d**2*s**2/3) * jnp.exp(-jnp.sqrt(5)*d*s)


def gauss(x, y, s):
    return jnp.exp(-(s ** 2) * _sqdist(x, y))   # paper eq. (22): exp(-sbar^2 ||x-x'||^2)


def kernel(kt, x, y, par, P):
    if kt == "szego":    return szego(x, y, par, P)
    if kt == "matern":   return matern52(x, y, par)
    if kt == "gaussian": return gauss(x, y, par)
    raise ValueError(kt)

In [ ]:
# ------------------- geometry helpers (uniform tensor grids) -------------------
def trapezoid_weights(coords):
    """Vertex weights of the trapezoidal rule on the uniform grid over [0,1]^2."""
    xs = np.unique(np.round(coords[:, 0], 12)); ys = np.unique(np.round(coords[:, 1], 12))
    wx = np.where((coords[:, 0] < 1e-12) | (coords[:, 0] > 1 - 1e-12), 0.5, 1.0)
    wy = np.where((coords[:, 1] < 1e-12) | (coords[:, 1] > 1 - 1e-12), 0.5, 1.0)
    return (xs[1] - xs[0]) * (ys[1] - ys[0]) * wx * wy


def bilinear_matrix(coarse_coords, fine_coords):
    """Sparse matrix W (n_fine x n_coarse): evaluates coarse-grid vertex values
    bilinearly at the fine nodes. Exact on bilinear functions, error O(h^2) --
    the same order as the P1 FEM (nearest neighbour would be O(h))."""
    n = int(round(1 / np.sort(np.unique(np.round(coarse_coords[:, 0], 12)))[1]))
    order = np.lexsort((coarse_coords[:, 0], coarse_coords[:, 1]))   # row-wise in y
    gx = np.clip(fine_coords[:, 0] * n, 0, n); gy = np.clip(fine_coords[:, 1] * n, 0, n)
    i0 = np.clip(np.floor(gx).astype(int), 0, n - 1); tx = gx - i0
    j0 = np.clip(np.floor(gy).astype(int), 0, n - 1); ty = gy - j0
    rows, cols, vals = [], [], []
    for di, dj, w in [(0, 0, (1-tx)*(1-ty)), (1, 0, tx*(1-ty)),
                      (0, 1, (1-tx)*ty),     (1, 1, tx*ty)]:
        rows.append(np.arange(len(fine_coords)))
        cols.append(order[(j0 + dj) * (n + 1) + (i0 + di)])
        vals.append(w)
    return csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
                      shape=(len(fine_coords), len(coarse_coords)))


def rel_l2(pred_fine, y_test, w_test, nrm_test):
    """Mean relative L2(Omega) error (trapezoidal rule) over the test samples."""
    return float((np.sqrt(np.maximum((y_test - pred_fine) ** 2 @ w_test, 0.0))
                  / nrm_test).mean())

In [ ]:
# ---------------- fit / CV / evaluation ----------------
def fit_predict(kt, par, nug, xtr, ytr, x_eval, P):
    """Fit on (xtr, ytr) with fixed (parameter, nugget) and predict at x_eval."""
    K = kernel(kt, xtr, xtr, par, P)
    K = K.at[jnp.diag_indices_from(K)].add(nug)
    alpha = cho_solve(cho_factor(K), ytr)          # representer theorem
    return np.asarray(kernel(kt, x_eval, xtr, par, P) @ alpha)


def eval_on_reference(pred_coarse, Wb, y_test, w_test, nrm_test):
    """Bilinear transfer to the reference mesh + relative L2 error."""
    return rel_l2((Wb @ pred_coarse.T).T, y_test, w_test, nrm_test)


def cv_fit_eval(kt, grid, nuggets, xtr, ytr, x_test, y_test, Wb,
                w_train, w_test, nrm_test, P, folds=5):
    """5-fold CV over grid x nuggets, then refit on the full pool and evaluate on
    the reference mesh. Returns (rel_err, (param, nugget))."""
    kf = list(KFold(folds, shuffle=True, random_state=0).split(np.asarray(xtr)))
    best, best_val = None, np.inf
    for par in grid:
        K_full = kernel(kt, xtr, xtr, par, P)      # built once, sliced below
        for nug in nuggets:
            fe = 0.0
            for tr, va in kf:
                K = K_full[tr][:, tr]
                K = K.at[jnp.diag_indices_from(K)].add(nug)
                alpha = cho_solve(cho_factor(K), ytr[tr])
                pred = K_full[va][:, tr] @ alpha
                fe += float(jnp.mean(jnp.sqrt(jnp.clip(
                    (ytr[va] - pred) ** 2 @ w_train, 0.0))))
            val = fe / len(kf)
            if np.isfinite(val) and val < best_val:
                best_val, best = val, (par, nug)
    par, nug = best
    pred_c = fit_predict(kt, par, nug, xtr, ytr, x_test, P)
    return eval_on_reference(pred_c, Wb, y_test, w_test, nrm_test), best

In [ ]:
# ---------------- N-experiment (P=10): CV-tuned kernels + pipeline floor ----------------
P = 10
# Shared grids (eq. (25) of the paper): the same sbar / c grids are used in the
# scalar experiments of Section 6.1, so that both experiments follow one protocol.
C_GRID = [0.01, 0.02, 0.05, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99]          # only 0.99 is theory-admissible
S_GRID = [0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100]      # = sbar of eqs. (22) and (24)
NUG = [1e-3, 1e-5, 1e-7, 1e-9]                          # = N * lambda of eq. (28)

ts = np.load('final_test_320.npz')
x_test = jnp.asarray(ts['x_test'])
y_test = np.asarray(ts['y_test'], dtype=np.float64)
w_test = trapezoid_weights(np.asarray(ts['coords']))
nrm = np.sqrt((y_test ** 2) @ w_test)

res_N = {}
for N in [100, 300, 500, 700, 900]:
    d = np.load(f'final320_train_N={N}.npz')
    coords_c = np.asarray(d['coords'])
    Wb = bilinear_matrix(coords_c, np.asarray(ts['coords']))
    w_train = jnp.asarray(trapezoid_weights(coords_c))
    ytc = np.asarray(d['y_test_coarse'], dtype=np.float64)
    res_N[f'{N}_floor'] = rel_l2((Wb @ ytc.T).T, y_test, w_test, nrm)
    print(f"== N={N}: pipeline floor = {res_N[f'{N}_floor']:.6f} ==", flush=True)
    for kt, grid in [("szego", C_GRID), ("matern", S_GRID), ("gaussian", S_GRID)]:
        t0, rels, pars = time.time(), [], []
        for rep in range(10):                      # 10 independent pools
            e, p = cv_fit_eval(kt, grid, NUG, jnp.asarray(d['x_pools'][rep]),
                               jnp.asarray(np.asarray(d['y_pools'][rep], np.float64)),
                               x_test, y_test, Wb, w_train, w_test, nrm, P)
            rels.append(e); pars.append(p)
        res_N[f'{N}_{kt}'] = np.array(rels)
        print(f'  {kt}: {np.mean(rels):.6f} +- {np.std(rels):.6f} '
              f'({time.time()-t0:.0f}s, params {sorted(set(p for p, _ in pars))}, '
              f'nuggets {sorted(set(g for _, g in pars))})', flush=True)
    np.savez('results_320.npz', **res_N)           # checkpoint after each N

In [ ]:
# ---------------- Figure 2: convergence + floor (P=10) ----------------
res_N = dict(np.load('results_320.npz'))
Ns = np.array([100, 300, 500, 700, 900])
STYLE = [("szego", "Szegö", "red"), ("matern", "Matérn 2.5", "orange"),
         ("gaussian", "RBF", "green")]

plt.figure(figsize=(9.5, 5.5))
plt.plot(Ns, [res_N[f'{N}_floor'] for N in Ns], '--', color='gray', lw=1.6,
         label="pipeline floor ($\\propto h^2 = e^{-N^{1/3}}$)")
for kt, name, c in STYLE:
    m = np.array([np.mean(res_N[f'{N}_{kt}']) for N in Ns])
    s = np.array([np.std(res_N[f'{N}_{kt}']) for N in Ns])
    slope, *_ = linregress(np.log(Ns), np.log(m))
    plt.plot(Ns, m, marker='o', color=c, lw=1.8, label=f"{name}: N^({slope:.2f})")
    plt.fill_between(Ns, m - s, m + s, color=c, alpha=0.2)
plt.yscale('log'); plt.xlabel("N"); plt.ylabel("Relative Test Error")
plt.xticks(Ns); plt.grid(True, which="both", ls=":", lw=0.5)
plt.legend(fontsize=9, title="Kernels")
plt.tight_layout(); plt.savefig("PDE-P=10-convergence-floor.pdf"); plt.show()

In [ ]:
# ---------------- Table 1: means +- stds with Welch significance tests (0.01) ----------------
def latex_table(res, labels, kts, names):
    lines = ["\\begin{tabular}{c|" + "c"*len(kts) + "|c}",
             "$N$ & " + " & ".join(names) + " & pipeline floor \\\\", "\\hline"]
    for lab in labels:
        errs = [res[f'{lab}_{k}'] for k in kts]
        means = [np.mean(e) for e in errs]; stds = [np.std(e) for e in errs]
        b = int(np.argmin(means))
        sig = stds[b] > 1e-12 and all(
            ttest_ind(errs[b], errs[j], equal_var=False)[1] <= 0.01
            for j in range(len(kts)) if j != b)
        row = []
        for k in range(len(kts)):
            d = max(0, -int(np.floor(np.log10(stds[k]))) + 1) if stds[k] > 0 else 6
            v = f"{means[k]:.{d}f} $\\pm$ {stds[k]:.{d}f}"
            row.append("\\textbf{" + v + "}" if (k == b and sig) else v)
        lines.append(f"{lab} & " + " & ".join(row) + f" & {res[f'{lab}_floor']:.6f} \\\\")
    lines.append("\\end{tabular}")
    return "\n".join(lines)

print(latex_table(res_N, [100, 300, 500, 700, 900],
                  ["szego", "matern", "gaussian"], ["Szegö", "Matérn", "Gaussian"]))   # column names as printed in Table 1

In [ ]:
# ---------------- Figure 3: parameter robustness (fixed kernel parameters) ----------------
# Szego with the admissible theory choice c_scale = 0.99 vs. Matern/Gaussian with the
# poorly scaled sbar = 1; only the nugget is selected by CV.
FIXED = {"szego": 0.99, "matern": 1.0, "gaussian": 1.0}

res_f3 = {}
for N in [100, 300, 500, 700, 900]:
    d = np.load(f'final320_train_N={N}.npz')
    coords_c = np.asarray(d['coords'])
    Wb = bilinear_matrix(coords_c, np.asarray(ts['coords']))
    w_train = jnp.asarray(trapezoid_weights(coords_c))
    for kt, par in FIXED.items():
        rels = []
        for rep in range(10):
            e, _ = cv_fit_eval(kt, [par], NUG, jnp.asarray(d['x_pools'][rep]),
                               jnp.asarray(np.asarray(d['y_pools'][rep], np.float64)),
                               x_test, y_test, Wb, w_train, w_test, nrm, P)
            rels.append(e)
        res_f3[f'{N}_{kt}'] = np.array(rels)
        print(f'N={N} {kt}: {np.mean(rels):.6f} +- {np.std(rels):.6f}', flush=True)
np.savez('results_fig3_320.npz', **res_f3)

plt.figure(figsize=(9.5, 5.5))
for kt, name, c in [("szego", "Szegö (admissible $c$)", "red"),
                    ("matern", "Matérn 2.5 ($\\bar s=1$)", "orange"),
                    ("gaussian", "RBF ($\\bar s=1$)", "green")]:
    m = np.array([np.mean(res_f3[f'{N}_{kt}']) for N in Ns])
    s = np.array([np.std(res_f3[f'{N}_{kt}']) for N in Ns])
    slope, *_ = linregress(np.log(Ns), np.log(m))
    plt.plot(Ns, m, marker='o', color=c, lw=1.8, label=f"{name}: N^({slope:.2f})")
    plt.fill_between(Ns, m - s, m + s, color=c, alpha=0.2)
plt.yscale('log'); plt.xlabel("N"); plt.ylabel("Relative Test Error")
plt.xticks(Ns); plt.grid(True, which="both", ls=":", lw=0.5)
plt.legend(fontsize=9, title="Kernels")
plt.tight_layout(); plt.savefig("PDE-P=10-robust-params.pdf"); plt.show()

In [ ]:
# ---------------- P-experiment (N=300): adapted Szego vs. isotropic kernels ----------------
C_GRID_P = [0.01, 0.02, 0.05, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99]   # shared grid, as above
S_GRID_P = [0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100]   # shared grid, as above   # = sbar

res_P = {}
for P_ in [1, 10, 30, 50]:
    d = np.load(f'pvar320_P={P_}.npz')
    x_te = jnp.asarray(d['x_test'])
    y_te = np.asarray(d['y_test'], dtype=np.float64)
    w_te = trapezoid_weights(np.asarray(d['coords_test']))
    nrm_ = np.sqrt((y_te ** 2) @ w_te)
    coords_c = np.asarray(d['coords_train'])
    Wb = bilinear_matrix(coords_c, np.asarray(d['coords_test']))
    w_tr = jnp.asarray(trapezoid_weights(coords_c))
    ytc = np.asarray(d['y_test_coarse'], dtype=np.float64)
    res_P[f'{P_}_floor'] = rel_l2((Wb @ ytc.T).T, y_te, w_te, nrm_)
    print(f"== P={P_}: pipeline floor = {res_P[f'{P_}_floor']:.6f} ==", flush=True)
    for kt, grid in [("szego", C_GRID_P), ("matern", S_GRID_P), ("gaussian", S_GRID_P)]:
        rels = []
        for rep in range(10):
            e, _ = cv_fit_eval(kt, grid, NUG, jnp.asarray(d['x_pools'][rep]),
                               jnp.asarray(np.asarray(d['y_pools'][rep], np.float64)),
                               x_te, y_te, Wb, w_tr, w_te, nrm_, P_)
            rels.append(e)
        res_P[f'{P_}_{kt}'] = np.array(rels)
        print(f'  {kt}: {np.mean(rels):.6f} +- {np.std(rels):.6f}', flush=True)
    np.savez('results_pvar320.npz', **res_P)       # checkpoint after each P

In [ ]:
# ---------------- Figure 4: varying P, fixed N=300 ----------------
res_P = dict(np.load('results_pvar320.npz'))
Ps = np.array([1, 10, 30, 50])

plt.figure(figsize=(9.5, 5.5))
plt.plot(Ps, [res_P[f'{P_}_floor'] for P_ in Ps], '--', color='gray', lw=1.6,
         label="pipeline floor")
for kt, name, c in [("szego", "Szegö ($c_p\\propto p^{-2/5}$)", "red"),
                    ("matern", "Matérn 2.5", "orange"), ("gaussian", "RBF", "green")]:
    m = np.array([np.mean(res_P[f'{P_}_{kt}']) for P_ in Ps])
    s = np.array([np.std(res_P[f'{P_}_{kt}']) for P_ in Ps])
    plt.plot(Ps, m, marker='o', ms=4, color=c, lw=1.8, label=name)
    plt.fill_between(Ps, np.maximum(m - s, m / 50), m + s, color=c, alpha=0.18)
plt.yscale('log'); plt.xlabel("Parameter dimension $P$")
plt.ylabel("Relative Test Error")
plt.xticks(Ps); plt.grid(True, which="both", ls=":", lw=0.5)
plt.legend(fontsize=9, title="Kernels")
plt.tight_layout(); plt.savefig("PDE-varying-P.pdf"); plt.show()

# cross-check of the factor quoted in Section 6.2 ("between 2.2 and 3.0")
print("\nfactor by which the isotropic kernels exceed the adapted Szego family:")
for P_ in [10, 30, 50]:
    s_ = np.mean(res_P[f'{P_}_szego'])
    print(f"  P={P_:2d}: Matern/Szego = {np.mean(res_P[f'{P_}_matern'])/s_:.2f}, "
          f"RBF/Szego = {np.mean(res_P[f'{P_}_gaussian'])/s_:.2f}")
print("\npipeline floor across P (Figure 4 caption: 'essentially constant in P'):")
print("  " + ", ".join(f"P={P_}: {res_P[f'{P_}_floor']:.6f}" for P_ in Ps))

## Test-set oracle sweep

Section 6.2 states:

> *A test-set oracle sweep over substantially denser grids confirms that this
> cross-validation attains the achievable optimum for every kernel, so that the
> differences reported below are not artifacts of model selection.*

The cell below reproduces that check. For every kernel it minimises the **test**
error over grids that are much denser than the CV grids (20 parameter values instead
of 6, 8 nuggets instead of 3) and compares the result with the CV-selected error of
the same repetitions. Two things have to hold for the claim in the paper:

1. the ratio CV / oracle is close to $1$ for **every** kernel, and
2. the ranking of the kernels is the same under CV and under the oracle.

The sweep is used only as a diagnostic; nothing in Figures 2–4 or Table 1 depends on
it. It is restricted to the first `REPS_ORACLE` repetitions to keep the runtime
manageable (the full $10\times$ sweep costs about a day on one CPU).

Requires the variables defined in the $N$-experiment cell above (`ts`, `x_test`,
`y_test`, `w_test`, `nrm`, `NUG`, `P`) and `results_320.npz`.

In [ ]:
# ---------------- test-set oracle sweep (claim in Section 6.2) ----------------
# Runtime: ~2-3 h on one CPU, dominated by N=900 (each fit costs ~2-3 s there).
# Set N_ORACLE = [300, 900] for a quick check; the conclusion does not change.
N_ORACLE = [100, 300, 500, 700, 900]
# The dense grids are constructed as supersets of the CV grids, so that the oracle
# error is <= the CV error by construction and the ratio below is always >= 1.
C_DENSE = sorted(set(C_GRID) | {round(v, 4) for v in np.linspace(0.99, 0.05, 20)})
S_DENSE = sorted(set(S_GRID) | {round(v, 5) for v in np.geomspace(0.2, 0.005, 20)})
NUG_DENSE = sorted(set(NUG) | {1e-3, 1e-5, 1e-7, 1e-9, 1e-10})
REPS_ORACLE = 3
print(f'dense grids: |C|={len(C_DENSE)}, |S|={len(S_DENSE)}, |nuggets|={len(NUG_DENSE)}')

# The CV reference is recomputed here for the same repetitions rather than read from
# results_320.npz: that file is written as a checkpoint inside the N-experiment loop,
# so an interrupted rerun can leave it holding only part of the sample sizes.
KTS = ["szego", "matern", "gaussian"]
CVGRID = {"szego": C_GRID, "matern": S_GRID, "gaussian": S_GRID}
res_or, res_cv = {}, {}

for N in N_ORACLE:
    d = np.load(f'final320_train_N={N}.npz')
    Wb = bilinear_matrix(np.asarray(d['coords']), np.asarray(ts['coords']))
    for kt, dense in [("szego", C_DENSE), ("matern", S_DENSE), ("gaussian", S_DENSE)]:
        t0, best_per_rep, argbest = time.time(), [], []
        for rep in range(REPS_ORACLE):
            xtr = jnp.asarray(d['x_pools'][rep])
            ytr = jnp.asarray(np.asarray(d['y_pools'][rep], np.float64))
            best, arg = np.inf, None
            for par in dense:
                for nug in NUG_DENSE:
                    try:
                        pred_c = fit_predict(kt, par, nug, xtr, ytr, x_test, P)
                        e = eval_on_reference(pred_c, Wb, y_test, w_test, nrm)
                    except Exception:
                        continue
                    if np.isfinite(e) and e < best:
                        best, arg = e, (par, nug)
            best_per_rep.append(best); argbest.append(arg)
        res_or[f'{N}_{kt}'] = np.array(best_per_rep)
        w_train = jnp.asarray(trapezoid_weights(np.asarray(d['coords'])))
        cvs = [cv_fit_eval(kt, CVGRID[kt], NUG, jnp.asarray(d['x_pools'][rep]),
                           jnp.asarray(np.asarray(d['y_pools'][rep], np.float64)),
                           x_test, y_test, Wb, w_train, w_test, nrm, P)[0]
               for rep in range(REPS_ORACLE)]
        res_cv[f'{N}_{kt}'] = np.array(cvs)
        cv, orc = float(np.mean(cvs)), float(np.mean(best_per_rep))
        print(f'N={N} {kt:8s}: oracle {orc:.6f}  CV {cv:.6f}  '
              f'CV/oracle {cv/orc:.3f}  best {argbest}  ({time.time()-t0:.0f}s)',
              flush=True)
    np.savez('results_oracle_320.npz', **res_or,
             **{f'cv_{k}': v for k, v in res_cv.items()})

print('\nranking of the kernels, CV vs oracle:')
for N in N_ORACLE:
    r_cv = sorted(KTS, key=lambda k: np.mean(res_cv[f'{N}_{k}']))
    r_or = sorted(KTS, key=lambda k: np.mean(res_or[f'{N}_{k}']))
    print(f'  N={N:4d}: CV {r_cv}  oracle {r_or}  '
          f'{"identical" if r_cv == r_or else "DIFFERENT -- check!"}')
print('\nworst CV/oracle ratio over all N and kernels: '
      f'{max(np.mean(res_cv[f"{N}_{k}"]) / np.mean(res_or[f"{N}_{k}"]) for N in N_ORACLE for k in KTS):.3f}')